In [2]:
import numpy as np
import pandas as pd
import time
import cv2
import matplotlib.pyplot as plt
import os
import json

In [3]:
def load_images_from_folder(folder):
    images = []
    for filename in os.listdir(folder):
        img = cv2.imread(os.path.join(folder,filename))
        if img is not None:
            images.append(img)
    return images

In [4]:
images1=load_images_from_folder("../InDir/NonWaterIm")
images2=load_images_from_folder("../InDir/WaterIm")

In [5]:
images1=np.array(images1)
images2=np.array(images2)

In [6]:
(winW, winH) = (48,48)

In [7]:
def sliding_window(image, stepSize, windowSize):
	# slide a window across the image
	for y in range(0, image.shape[0], stepSize):
		for x in range(0, image.shape[1], stepSize):
#			print(y + windowSize[1])
			yield (x, y, image[y:y + windowSize[1], x:x + windowSize[0]])

In [8]:
def func(images):
    im=[]
    count=0
    for img1 in images:
        print(count)
        count=count+1;
        for (x, y, window) in sliding_window(img1, stepSize=32, windowSize=(winW, winH)):
            if window.shape[0] != winH or window.shape[1] != winW:
                continue
            #clone = img1.copy()
            #cv2.rectangle(clone, (x, y), (x + winW, y + winH), (0, 255, 0), 2)
            #x=cv2.resize(window, (224,224)) 
            im.append(window)
            #cv2.imshow("clone", clone)
            #cv2.imwrite(f'../InDir/train64/{i}.jpg',window)
            #cv2.waitKey(1)
            #time.sleep(0.025)
        #cv2.waitKey(0)
        #cv2.destroyAllWindows()
    return im

In [9]:
im1=func(images1)
im2=func(images2)
im1=np.asarray(im1)
im2=np.asarray(im2)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [13]:
im=np.concatenate((im1,im2),axis=0)
im = im.astype('float32')
im/=255

In [14]:
y=np.zeros(len(im))
y[len(im1):]=1

In [15]:
y.shape

(54709,)

In [16]:
from sklearn.preprocessing import LabelEncoder
from keras.utils import np_utils
encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)
y= np_utils.to_categorical(y)

Using TensorFlow backend.


In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(im, y, test_size=0.4, random_state=0)#train test random split

In [19]:
X_train.shape

(32825, 48, 48, 3)

In [73]:
def identity_block(input_tensor, kernel_size, filters):
    L2_WEIGHT_DECAY=0.01
    BATCH_NORM_DECAY=0.99
    """The identity block is the block that has no conv layer at shortcut.
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of
            middle conv layer at main path
        filters: list of integers, the filters of 3 conv layer at main path
    """
    filters1, filters2, filters3 = filters
    bn_axis = 3
 
    x = layers.Conv2D(filters1, (1, 1), use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(input_tensor)
 
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY)(x)
    x = layers.Activation('relu')(x)
 
    x = layers.Conv2D(filters2, kernel_size,
                      padding='same', use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
 
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY)(x)
 
    x = layers.Activation('relu')(x)
 
    x = layers.Conv2D(filters3, (1, 1), use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
 
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY)(x)
 
    x = layers.add([x, input_tensor])
    x = layers.Activation('relu')(x)
    return x

In [74]:
def conv_block(input_tensor, kernel_size, filters, strides=(2, 2)):
    L2_WEIGHT_DECAY=0.01
    BATCH_NORM_DECAY=0.99
    """A block that has a conv layer at shortcut.
    # Arguments
        input_tensor: input tensor
        kernel_size: default 3, the kernel size of
            middle conv layer at main path
        filters: list of integers, the filters of 3 conv layer at main path
        stage: integer, current stage label, used for generating layer names
    # Returns
        Output tensor for the block.
    Note that from stage 3,
    the second conv layer at main path is with strides=(2, 2)
    And the shortcut should have strides=(2, 2) as well
    """
 
    filters1, filters2, filters3 = filters

    bn_axis = 3
 
    x = layers.Conv2D(filters1, (1, 1), use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(input_tensor)
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY)(x)
    x = layers.Activation('relu')(x)
 
 
    x = layers.Conv2D(filters2, kernel_size, strides=strides, padding='same',
                      use_bias=False, kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY)(x)
    x = layers.Activation('relu')(x)
 
    x = layers.Conv2D(filters3, (1, 1), use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY)(x)
 
    shortcut = layers.Conv2D(filters3, (1, 1), strides=strides, use_bias=False,
                             kernel_initializer='he_normal',
                             kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(input_tensor)
    shortcut = layers.BatchNormalization(axis=bn_axis,
                                         momentum=BATCH_NORM_DECAY)(shortcut)
 
    x = layers.add([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

In [75]:
def resnet50(num_classes, input_shape):
    L2_WEIGHT_DECAY=0.01
    BATCH_NORM_DECAY=0.99
    
    img_input = layers.Input(shape=input_shape)
 

    x = img_input
    bn_axis = 3
 
    # Conv1 (7x7,64,stride=2)
    x = layers.ZeroPadding2D(padding=(3, 3))(x)
 
    x = layers.Conv2D(64, (7, 7),
                      strides=(2, 2),
                      padding='valid', use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY,
                                  )(x)
    x = layers.Activation('relu')(x)
    x = layers.ZeroPadding2D(padding=(1, 1))(x)
 
    # 3x3 max pool,stride=2
    x = layers.MaxPooling2D((3, 3), strides=(2, 2))(x)
 
    # Conv2_x
 
    # 1×1, 64
    # 3×3, 64
    # 1×1, 256
 
    x = conv_block(x, 3, [64, 64, 256], strides=(1, 1))
    x = identity_block(x, 3, [64, 64, 256])
    x = identity_block(x, 3, [64, 64, 256])
 

    # Conv4_x
    # 1×1, 256
    # 3×3, 256
    # 1×1, 1024
    x = conv_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
 
 
    # average pool, 1000-d fc, softmax
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        num_classes, activation='softmax',
        kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY),
        bias_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
 
    # Create model.
    return models.Model(img_input, x, name='resnet50')

In [76]:
from keras import layers
from keras import regularizers
from keras import models
model =resnet50(2,X_train.shape[1:])

In [77]:
model.summary()

__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_15 (InputLayer)           (None, 48, 48, 3)    0                                            
__________________________________________________________________________________________________
zero_padding2d_20 (ZeroPadding2 (None, 54, 54, 3)    0           input_15[0][0]                   
__________________________________________________________________________________________________
conv2d_141 (Conv2D)             (None, 24, 24, 64)   9408        zero_padding2d_20[0][0]          
__________________________________________________________________________________________________
batch_normalization_136 (BatchN (None, 24, 24, 64)   256         conv2d_141[0][0]                 
__________________________________________________________________________________________________
activation

In [78]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [79]:
model.fit(X_train,Y_train, batch_size=32, epochs=5, validation_split=0.3)

Train on 22977 samples, validate on 9848 samples
Epoch 1/5
22977/22977 [==============================] - 932s 41ms/step - loss: 16.0327 - acc: 0.9298 - val_loss: 0.8736 - val_acc: 0.9055
Epoch 2/5
22977/22977 [==============================] - 733s 32ms/step - loss: 0.5656 - acc: 0.9384 - val_loss: 0.7708 - val_acc: 0.8826
Epoch 3/5
22977/22977 [==============================] - 787s 34ms/step - loss: 0.5295 - acc: 0.9387 - val_loss: 0.5096 - val_acc: 0.8956
Epoch 4/5
22977/22977 [==============================] - 769s 33ms/step - loss: 0.5025 - acc: 0.9401 - val_loss: 3.4741 - val_acc: 0.8823
Epoch 5/5
22977/22977 [==============================] - 778s 34ms/step - loss: 0.4764 - acc: 0.9418 - val_loss: 0.8555 - val_acc: 0.8824


In [88]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix
Y_test2=np.argmax(Y_test,axis=1)
Y_pred=model.predict(X_test)
#Y_pred=model.predict_proba(X_test)
y_pred = np.argmax(Y_pred, axis=1)
acc=accuracy_score(Y_test2, y_pred)
print(acc)
print(confusion_matrix(Y_test2,y_pred))
print(classification_report(Y_test2,y_pred))

0.8839334673734235
[[19344     0]
 [ 2540     0]]
              precision    recall  f1-score   support

           0       0.88      1.00      0.94     19344
           1       0.00      0.00      0.00      2540

    accuracy                           0.88     21884
   macro avg       0.44      0.50      0.47     21884
weighted avg       0.78      0.88      0.83     21884



C:\Users\user\.conda\envs\tcs\lib\site-packages\sklearn\metrics\classification.py:1437: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


In [80]:
model.save('../models/model_resnet1.h5')

In [ ]:
from tensorflow.keras.models import load_model
model = load_model('../models/model2.h5', compile=False)
model.load_weights('../models/model2.h5')

In [81]:
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import CustomObjectScope
from tensorflow.keras.initializers import glorot_uniform
with CustomObjectScope({'GlorotUniform': glorot_uniform()}):
        model = load_model('../models/model_resnet1.h5')

Instructions for updating:
`normal` is a deprecated alias for `truncated_normal`


In [82]:
img1=cv2.imread('../test/11.jpg',cv2.IMREAD_COLOR);
#img1 = img1.astype('float32')
#img1/=255
#img1 = cv2.resize(img1, (960, 540))  
#img1 = cv2.resize(img1, (1024,1024), interpolation=cv2.INTER_AREA)
(winW, winH) = (48,48)

In [83]:
def sliding_window(image, stepSize, windowSize):
	# slide a window across the image
	for y in range(0, image.shape[0], stepSize):
		for x in range(0, image.shape[1], stepSize):
#			print(y + windowSize[1])
			yield (x, y, image[y:y + windowSize[1], x:x + windowSize[0]])

In [84]:
img1.shape

(534, 800, 3)

In [85]:
steps=48
img1= np.pad(img1, pad_width=steps, mode='constant', constant_values=0)
img1=img1[:,:,steps:steps+3]
mask=np.zeros(np.asarray(img1).shape[:2])
mask.shape

(630, 896)

In [86]:
for (x, y, window) in sliding_window(img1, stepSize=8, windowSize=(winW, winH)):
    if window.shape[0] != winH or window.shape[1] != winW:
        continue
    X= np.expand_dims(window, axis=0)
    Y_pred=model.predict(X)
    y_pred = np.argmax(Y_pred, axis=1)
    
    mask[y:y + winW, x:x + winH]+=np.full((48,48), y_pred)
    
    clone = img1.copy()
    val=int(255*y_pred[0])
    cv2.rectangle(clone, (x, y), (x + winW, y + winH), (0, 255,val), 2)
    cv2.imshow("clone", clone)
    #cv2.imshow("window",window)
    cv2.waitKey(1)
    #time.sleep(0.015)
cv2.waitKey(0)
cv2.destroyAllWindows()

KeyboardInterrupt: 

In [ ]:
Y_pred

In [ ]:
img1=img1[steps:-steps,steps:-steps]
mask=mask[steps:-steps,steps:-steps]

In [ ]:
maxval = np.amax(mask)
maxval

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
mask3= scaler.fit_transform(mask)
mask3*=255

In [ ]:
mask2=mask/maxval

In [ ]:
for i in range(img1.shape[:1][0]):
    for j in range(img1.shape[1:2][0]):
        if(mask2[i,j]>=0.0002):
            mask2[i][j]=255
        else:
            mask2[i,j]=0

In [ ]:
cv2.imshow("mask", mask2)
cv2.waitKey(0)
#cv2.imshow("mask", thresh1)
#cv2.waitKey(0)
cv2.imshow("mask", mask3)
cv2.waitKey(0)
cv2.imshow("img",img1)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
def resnet50(num_classes, input_shape):
    L2_WEIGHT_DECAY=0.01
    BATCH_NORM_DECAY=0.99
    
    img_input = layers.Input(shape=input_shape)
 

    x = img_input
    bn_axis = 3
 
    # Conv1 (7x7,64,stride=2)
    x = layers.ZeroPadding2D(padding=(3, 3))(x)
 
    x = layers.Conv2D(64, (7, 7),
                      strides=(2, 2),
                      padding='valid', use_bias=False,
                      kernel_initializer='he_normal',
                      kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
    x = layers.BatchNormalization(axis=bn_axis,
                                  momentum=BATCH_NORM_DECAY,
                                  )(x)
    x = layers.Activation('relu')(x)
    x = layers.ZeroPadding2D(padding=(1, 1))(x)
 
    # 3x3 max pool,stride=2
    x = layers.MaxPooling2D((3, 3), strides=(2, 2))(x)
 
    # Conv2_x
 
    # 1×1, 64
    # 3×3, 64
    # 1×1, 256
 
    x = conv_block(x, 3, [64, 64, 256], strides=(1, 1))
    x = identity_block(x, 3, [64, 64, 256])
    x = identity_block(x, 3, [64, 64, 256])
 
    # Conv3_x
    #
    # 1×1, 128
    # 3×3, 128
    # 1×1, 512
 
    x = conv_block(x, 3, [128, 128, 512])
    x = identity_block(x, 3, [128, 128, 512])
    x = identity_block(x, 3, [128, 128, 512])
    x = identity_block(x, 3, [128, 128, 512])
 
    # Conv4_x
    # 1×1, 256
    # 3×3, 256
    # 1×1, 1024
    x = conv_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
    x = identity_block(x, 3, [256, 256, 1024])
 
    # 1×1, 512
    # 3×3, 512
    # 1×1, 2048
    x = conv_block(x, 3, [512, 512, 2048])
    x = identity_block(x, 3, [512, 512, 2048])
    x = identity_block(x, 3, [512, 512, 2048])
 
    # average pool, 1000-d fc, softmax
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        num_classes, activation='softmax',
        kernel_regularizer=regularizers.l2(L2_WEIGHT_DECAY),
        bias_regularizer=regularizers.l2(L2_WEIGHT_DECAY))(x)
 
    # Create model.
    return models.Model(img_input, x, name='resnet50')